In [4]:
print(123)

123


In [5]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [6]:
from openai import OpenAI
import os
openai_client = OpenAI(
    api_key=os.getenv("OPEN_API_KEY")
)

In [7]:
def llm(prompt):
    response = openai_client.responses.create(
        model = 'gpt-5.4-mini',
        input = prompt
    )
    return response.output_text

In [8]:
question="I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely — in many cases you can still join after a course has started.

A few things determine whether it’s possible:
- whether enrollment is still open
- how far along the course is
- whether there are prerequisites or limited seats
- whether you’ll have access to any missed material

If you want, send me the course name or link, and I can help you figure out whether it’s still joinable and what to ask the instructor or organizer.


In [9]:
context = '''
General Course-Related Questions
#
I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.
edit on GitHub
#
Cloud alternatives with GPU

Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

    Google Colab
    Kaggle
    Databricks (possibly)

Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
edit on GitHub
#
Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?

When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

    First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
    Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!

'''

In [10]:
prompt = f'''
your taks is to answer the question based on the following context.
Use the context to find relevant information to answer the question and provide accurate answers.
If the answer is not found in the context, respond with "I don't know".

Question: {question}

Context: {context}
'''

In [11]:
print(prompt)


your taks is to answer the question based on the following context.
Use the context to find relevant information to answer the question and provide accurate answers.
If the answer is not found in the context, respond with "I don't know".

Question: I just discovered the course. Can I join now?

Context: 
General Course-Related Questions
#
I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom lin

In [12]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [13]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [14]:
import requests
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()
courses_raw[:2]

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93}]

In [15]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f'''{url_prefix}/{course['path']}'''
    
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
len(documents)

1350

In [16]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [17]:
from minsearch import Index
index = Index(
    text_fields=['question','section','answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [18]:
import pandas as pd
pd.DataFrame(documents)['course'].unique()

<StringArray>
[       'data-engineering-zoomcamp', 'stock-markets-analytics-zoomcamp',
            'ai-dev-tools-zoomcamp',                     'llm-zoomcamp',
                   'mlops-zoomcamp',        'machine-learning-zoomcamp']
Length: 6, dtype: str

In [19]:
question = 'I just discovered the course. Can I join now?'
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question':2, 'section':0.5}
    filter_dict = {'course':course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results =5
    )


In [20]:
search_results = search(question, course='llm-zoomcamp')
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [21]:
INSTRUCTIONS = '''
Your task is to answer the questions from the course partipants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context,
respond with "I don't know".
'''

In [22]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_context(search_results):
    lines = []
    
    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q:" + doc['question'])
        lines.append("A: " + doc['answer'])
        lines.append("")
    
    return "\n".join(lines).strip()

In [24]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question,
        context = context
    )
    return prompt.strip()

In [25]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q:I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q:Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q:Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.



In [26]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = prompt
)

In [32]:
response.output[0]

ResponseOutputMessage(id='msg_07d18ff6493d9b88006a3ba68316a4819e94a02b51aea48a03', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now and start learning.\n\nIf you want to receive a certificate, though, you need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [43]:
response.output[0].content[0].text

'Yes — you can still join now and start learning.\n\nIf you want to receive a certificate, though, you need to submit your project while submissions are still open.'

In [46]:
type(response.output[0])

openai.types.responses.response_output_message.ResponseOutputMessage

In [47]:
response.usage

ResponseUsage(input_tokens=479, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=37, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=516)

In [49]:
input_price = 0.75/1_000_000
output_price = 4.5/1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0005257500000000001

In [50]:
message_history = [
    {'role': 'developer',
     'content': INSTRUCTIONS},
     {'role': 'user', 'content': prompt}
]

response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = message_history
)

In [51]:
def llm(instructions, user_prompt, model = 'gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create(
        model = model,
        input = message_history
    )

    return response.output_text

In [57]:
def rag(query, model = 'gpt-5.4-mini'):
    search_results = search(query)
    print(search_results)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model = model)
    return answer

In [55]:
answer = rag('I just discovered the course. Can I join now?')
print(answer)

Yes, you can join now. If you want a certificate, though, you need to submit your project while submissions are still open.


In [59]:
rag('How do I get the certificate from the course data engineering')

[{'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'}, {'id': '9f689c185f', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I missed the first homework - can I still get a certificate?', 'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'}, {'id': '04919992b3', 'course': 'llm-zoomca

'To get the certificate for the Data Engineering course, you need to finish the course with a **live cohort** and **pass the Capstone project**.\n\nA few important notes:\n- **Self-paced mode does not qualify** for a certificate.\n- You also need to **peer-review 3 capstone projects** after submitting yours.\n- If you missed some homework, that’s okay — **homework is not mandatory** for the certificate.\n\nAlso, make sure your **official name** is filled in your course profile, since that name will appear on the certificate.'

In [63]:
c1= response.usage.input_tokens
c2= response.usage.output_tokens
cost = input_price*c1 + c2*output_price 
cost

0.000534